# Chạy Thực Nghiệm Đồ Án (Continual Learning)
Notebook này được thiết kế ĐỘC QUYỀN để chạy toàn bộ file `run_experiments.sh` tự động trên Kaggle.

So sánh phương pháp đề xuất **Coreset (Bilevel Optimization)** với 4 baseline: Uniform, KMeans, **Sensitivity Coreset**, và **GLISTER**.

`run_experiments.sh` được tách thành 2 giai đoạn độc lập qua tham số `--stage`, để không phải huấn luyện lại chỉ vì muốn sửa/vẽ lại biểu đồ:
- **`--stage train`** (Ô Số 2): chạy huấn luyện, cần GPU + torch/jax, tốn 2-3 tiếng. Chỉ ghi ra các file kết quả `.txt` (metric, không phải trọng số mô hình) trong `cl_streaming/cl_results/`.
- **`--stage report`** (Ô Số 2b): chỉ đọc các file `.txt` đã có để tổng hợp bảng số liệu và vẽ biểu đồ, chạy trong vài giây, KHÔNG cần GPU/torch/jax (chỉ cần numpy/matplotlib/seaborn). Có thể chạy lại nhiều lần, ở phiên Kaggle khác, hoặc thậm chí trên máy tính cá nhân sau khi tải `cl_results/` về.

**HƯỚNG DẪN:**
1. Bật GPU (T4 x2 hoặc P100) trong Settings của Kaggle.
2. Chạy Ô Số 1 để cài đặt môi trường.
3. Bấm **Restart Session / Restart Kernel** khi Kaggle yêu cầu.
4. Chạy Ô Số 2 để huấn luyện (`--stage train`, tốn khoảng 2-3 tiếng, cứ treo máy để đó). Sau khi xong, tải thư mục `cl_streaming/cl_results/` về máy để lưu lại.
5. Chạy Ô Số 2b (`--stage report`) để tổng hợp bảng số liệu + vẽ biểu đồ — có thể chạy lại bất cứ lúc nào mà không cần lặp lại bước 4. Nếu quay lại ở phiên làm việc mới (session mới, không còn `cl_results/` cũ), hãy tải thư mục `cl_results/` đã lưu lên lại đúng đường dẫn `cl_streaming/cl_results/` trước khi chạy ô này.
6. Chạy Ô Số 3 để xem các biểu đồ so sánh trực quan ngay trong notebook.

In [ ]:
# Ô SỐ 1: CÀI ĐẶT MÔI TRƯỜNG VÀ TẢI MÃ NGUỒN
!git clone https://github.com/quachthanhhmd/bilevel-coresets.git
%cd bilevel-coresets
!pip install neural-tangents

⚠️ **CẢNH BÁO QUAN TRỌNG:** Dừng lại tại đây! Bạn phải bấm `Restart Session` (hoặc `Restart Kernel`) trước khi chạy ô tiếp theo.

In [ ]:
# Ô SỐ 2: HUẤN LUYỆN (nặng, cần GPU) — chỉ ghi ra cl_results/*.txt, KHÔNG vẽ biểu đồ
# Lưu ý: Lệnh %cd giúp đảm bảo thư mục làm việc luôn nằm trong repo sau khi restart kernel.
%cd /kaggle/working/bilevel-coresets

# Chấp quyền thực thi cho file bash
!chmod +x run_experiments.sh

# Khởi chạy thực nghiệm trên tập FashionMNIST, so sánh Uniform, KMeans, Sensitivity Coreset, GLISTER và Coreset (Bilevel)
!./run_experiments.sh --dataset splitfashionmnist --methods uniform,kmeans_features,sensitivity,glister,coreset --stage train

🎉 **LẤY KẾT QUẢ:** Sau khi ô số 2 chạy xong (khoảng vài tiếng), các file kết quả chi tiết `.txt` (chứa metric dạng JSON — test accuracy, ma trận accuracy theo task, thời gian chạy; KHÔNG phải trọng số mô hình) sẽ nằm trong thư mục `cl_streaming/cl_results/`. Hãy tải thư mục này về máy để không phải huấn luyện lại nếu cần vẽ lại biểu đồ sau này (kể cả ở phiên Kaggle khác hoặc trên máy cá nhân).

## Ô SỐ 2b: Tổng hợp số liệu + Vẽ biểu đồ (nhẹ, không cần GPU)
Chỉ đọc các file `cl_streaming/cl_results/*.txt` đã có sẵn để in bảng số liệu và sinh các file `.png` trong `experiments/`. Có thể chạy lại nhiều lần bất cứ khi nào muốn chỉnh/vẽ lại biểu đồ, mà không cần chạy lại Ô Số 2.

Nếu bạn quay lại ở một phiên Kaggle/Colab mới (đã restart, không còn `cl_results/` cũ trong bộ nhớ), hãy tải thư mục `cl_results/` đã lưu ở bước trên lên lại đúng đường dẫn `cl_streaming/cl_results/` trước khi chạy ô này.

In [ ]:
%cd /kaggle/working/bilevel-coresets
!./run_experiments.sh --dataset splitfashionmnist --methods uniform,kmeans_features,sensitivity,glister,coreset --stage report

## Ô SỐ 3: Trực quan hóa kết quả so sánh với baseline
Hiển thị lại các biểu đồ đã được sinh ra ở Ô Số 2b (so sánh Coreset (Bilevel) với Sensitivity Coreset và GLISTER).

In [ ]:
from IPython.display import Image, display

for img_name in [
    'average_forgetting.png',
    'tradeoff_time_accuracy.png',
    'baseline_accuracy_comparison.png',
    'baseline_forgetting_comparison.png',
    'baseline_tradeoff_comparison.png',
]:
    img_path = f'experiments/{img_name}'
    print(f'--- {img_name} ---')
    display(Image(filename=img_path))